# 实验4 复杂网络的社团检测

1. 给定网络 `1e0w300.csv`，采用 3 种课件算法进行社团检测，输出模块度并对社团划分后的网络进行可视化。
2. 对教材 P144 图 4-19(a) 的网络，参考 `networkx_reference3.0.pdf`，采用派系过滤算法进行社团检测。

## 第一题：1e0w300.csv 的 3 种社团检测算法

In [ ]:
# 读取网络数据
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import networkx.algorithms.community as nx_comm
from networkx.algorithms.community import greedy_modularity_communities
from random import randint

protein = pd.read_csv('1e0w300.csv')
G = nx.from_pandas_edgelist(protein, 'source', 'target', edge_attr='weight')

print('网络节点数为：', G.number_of_nodes())
print('网络边数为：', G.number_of_edges())

pos = nx.spring_layout(G, seed=10, weight='weight')
node_list = list(G.nodes())
node_index = {node: index for index, node in enumerate(node_list)}


In [ ]:
# greedy modularity communities 方法（CNM算法）
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities
import matplotlib.pyplot as plt
from random import randint

plt.figure(figsize=(12, 5))
plt.subplot(121)
nx.draw(G, pos, node_size=18, with_labels=False, width=0.2)
plt.title('1e0w300 Graph')

# 使用 greedy modularity（CNM）算法进行社团检测
partitions = greedy_modularity_communities(G, weight='weight')
print('网络社团为：', partitions)

# 计算模块度
Q = nx.community.modularity(G, partitions, weight='weight')
print('模块度Q=', Q)

colors = ['' for x in range(G.number_of_nodes())]
counter = 0
for partition in partitions:
    color = '#%06X' % randint(0, 0xFFFFFF)
    counter += 1

    for node in partition:
        colors[node_index[node]] = color

print('社团数量为：', counter)
plt.axis('off')
plt.subplot(122)
nx.draw_networkx(G, pos, nodelist=node_list, node_size=18, with_labels=False, width=0.2, node_color=colors)
plt.title('1e0w300 Graph with CNM Communities')
plt.show()


In [ ]:
# Louvain 算法
import networkx as nx
import networkx.algorithms.community as nx_comm
import matplotlib.pyplot as plt
from random import randint

plt.figure(figsize=(12, 5))
plt.subplot(121)
nx.draw(G, pos, node_size=18, with_labels=False, width=0.2)
plt.title('1e0w300 Graph')

# 使用 Louvain 算法进行社团检测
partition = nx_comm.louvain_communities(G, weight='weight', seed=10)
print('网络社团为：', partition)

# 计算模块度
Q = nx_comm.modularity(G, partition, weight='weight')
print('模块度Q=', Q)

# 初始化颜色列表，用于存储每个节点的颜色
colors = ['' for _ in range(G.number_of_nodes())]

# 为每个社团分配随机颜色
for com in partition:
    # 生成一个随机的十六进制颜色代码（RGB）
    color = '#%06X' % randint(0, 0xFFFFFF)
    # 为当前社团的所有节点分配相同的颜色
    for node in com:
        colors[node_index[node]] = color

print('社团数量：', len(partition))
plt.axis('off')
plt.subplot(122)
nx.draw_networkx(G, pos, nodelist=node_list, node_size=18, with_labels=False, width=0.2, node_color=colors)
plt.title('1e0w300 Graph with Louvain Communities')
plt.show()


In [ ]:
# 利用标签传播算法进行社团检测
# 参考论文：Near linear time algorithm to detect community structures in large-scale networks
import networkx as nx
import networkx.algorithms.community as nx_comm
import matplotlib.pyplot as plt
from random import randint

plt.figure(figsize=(12, 5))
plt.subplot(121)
nx.draw(G, pos, node_size=18, with_labels=False, width=0.2)
plt.title('1e0w300 Graph')

# 标签传播算法进行社团检测
partition = list(nx.community.label_propagation_communities(G))
print('网络社团为：', partition)

Q = nx_comm.modularity(G, partition, weight='weight')
print('模块度Q=', Q)

colors = ['' for x in range(G.number_of_nodes())]
counter = 0
for com in partition:
    color = '#%06X' % randint(0, 0xFFFFFF)
    counter += 1
    for node in list(com):
        colors[node_index[node]] = color

print('社团数量为：', len(partition))
plt.axis('off')
plt.subplot(122)
nx.draw_networkx(G, pos, nodelist=node_list, node_size=18, with_labels=False, width=0.2, node_color=colors)
plt.title('1e0w300 Graph with Label Propagation Communities')
plt.show()


### 三种算法结果分析

1. CNM 算法以模块度贪心增大为目标，划分结果通常比较稳定，得到的社团结构较清晰，适合做基础社团检测实验。
2. Louvain 算法也是基于模块度优化，但采用了多层聚合思想，通常能得到更高的模块度，且在较大网络上运行速度较快。
3. 标签传播算法不直接优化模块度，而是通过节点标签在网络中传播形成社团，因此运行速度快，但结果稳定性相对较弱，划分出的社团数量往往更多。
4. 在本实验网络上，Louvain 算法得到的模块度通常最高，CNM 次之，标签传播算法的模块度相对较低，因此从社团划分质量看，Louvain 效果最好。

## 第二题：教材 P144 图 4-19(a) 的派系过滤算法

按照教材图 4-19(a) 手工构造网络，再使用 `networkx.algorithms.community.k_clique_communities` 进行 4-派系社团检测。

In [ ]:
# 派系过滤算法（k-clique communities）
import networkx as nx
import matplotlib.pyplot as plt
from random import randint
from networkx.algorithms.community import k_clique_communities

G = nx.Graph()

# 按教材图 4-19(a) 手工构造网络
G.add_edges_from([
    (0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3),
    (4, 5), (4, 6), (4, 7), (4, 8), (5, 6), (5, 7), (5, 8), (6, 7), (6, 8), (7, 8),
    (4, 9), (5, 9), (6, 9),
    (4, 10), (7, 10), (8, 10),
    (2, 4)
])

pos = {
    0: (0, 2), 1: (1, 2), 2: (0, 1), 3: (1, 1),
    4: (3, 1), 5: (4, 2), 6: (4, 0), 7: (5, 1.5), 8: (5, 0.5), 9: (6.5, 1.2), 10: (2, 0.2)
}

plt.figure(figsize=(12, 5))
plt.subplot(121)
nx.draw(G, pos, node_size=400, with_labels=True, width=1.2)
plt.title('Figure 4-19(a) Graph')

cliques = list(nx.find_cliques(G))
print('极大派系为：', cliques)

# 使用派系过滤算法进行 4-派系社团检测
partition = list(k_clique_communities(G, 4, cliques))
print('4-派系社团为：', partition)

colors = ['' for x in range(G.number_of_nodes())]
for com in partition:
    color = '#%06X' % randint(0, 0xFFFFFF)
    for node in com:
        colors[node] = color

plt.axis('off')
plt.subplot(122)
nx.draw_networkx(G, pos, node_size=400, with_labels=True, width=1.2, node_color=colors)
plt.title('4-Clique Communities')
plt.show()


### 第二题结果说明

1. `k_clique_communities(G, 4)` 表示寻找 4-派系社团，即由大小为 4 的派系通过共享 3 个节点连接而成的社团。
2. 该网络中可以得到 2 个 4-派系社团，其中一个为左侧的 4 节点完全子图，另一个为右侧由多个重叠 4-派系合并得到的大社团。
3. 派系过滤算法的特点是能够识别重叠紧密子结构，适合发现网络中的高凝聚社团。